# PMLS Assignment 1 - Model Deployment Using FastAPI

Working out which customer segments responded best to a skin clinic campaign aimed at 10,000 customers. I calculate response rates for four segmentations here, then move the same logic into a FastAPI app.

In [1]:
import pandas as pd

## Load and check the data

In [2]:
df = pd.read_csv("skin clinic campaign.csv")
df.head()

,CustID,Gender,AgeGroup,Purchase_Last_Quarter,Unique_Products_Purchased,Response_to_Campaign
0,100001,Female,30-50,Yes,9,Yes
1,100002,Male,>50,Yes,15,No
2,100003,Male,>50,Yes,11,No
3,100004,Female,30-50,No,6,No
4,100005,Female,<30,Yes,13,Yes


In [3]:
print("shape:", df.shape)
df.info()

shape: (10000, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   CustID                     10000 non-null  int64 
 1   Gender                     10000 non-null  object
 2   AgeGroup                   10000 non-null  object
 3   Purchase_Last_Quarter      10000 non-null  object
 4   Unique_Products_Purchased  10000 non-null  int64 
 5   Response_to_Campaign       10000 non-null  object
dtypes: int64(2), object(4)
memory usage: 468.9+ KB


In [4]:
print("missing values:")
print(df.isnull().sum())

missing values:
CustID                       0
Gender                       0
AgeGroup                     0
Purchase_Last_Quarter        0
Unique_Products_Purchased    0
Response_to_Campaign         0
dtype: int64


In [5]:
# check what is actually in each column before I group on it
for c in ["Gender", "AgeGroup", "Purchase_Last_Quarter", "Response_to_Campaign"]:
    print(f"{c}: {sorted(df[c].unique())}")

print("\nUnique_Products_Purchased range:",
      df["Unique_Products_Purchased"].min(), "to", df["Unique_Products_Purchased"].max())

Gender: ['Female', 'Male']
AgeGroup: ['30-50', '<30', '>50']
Purchase_Last_Quarter: ['No', 'Yes']
Response_to_Campaign: ['No', 'Yes']

Unique_Products_Purchased range: 1 to 15


In [6]:
print("overall response rate:", round(df["Response_to_Campaign"].eq("Yes").mean() * 100, 2), "%")

overall response rate: 39.9 %


`AgeGroup` already holds `<30`, `30-50` and `>50`, so I use it as it is rather than
re-binning anything. No missing values and no odd categories, so the data is ready to
group on.

In [7]:
# map the target to 1/0 so the mean of the column is the response rate
df["Responded"] = df["Response_to_Campaign"].map({"Yes": 1, "No": 0})

# AgeGroup would sort alphabetically and put 30-50 first, so set the real order
df["AgeGroup"] = pd.Categorical(df["AgeGroup"], categories=["<30", "30-50", ">50"], ordered=True)

# bin the product count into the three groups the assignment asks for
df["ProductGroup"] = pd.cut(df["Unique_Products_Purchased"],
                            bins=[0, 4, 8, float("inf")],
                            labels=["1-4", "5-8", ">8"])

print(df["ProductGroup"].value_counts().sort_index())

ProductGroup
1-4    2091
5-8    3703
>8     4206
Name: count, dtype: int64


In [8]:
# one function for all four tables
def response_table(df, group_col):
    out = df.groupby(group_col, observed=True).agg(
        Customers=("Responded", "size"),
        Responded=("Responded", "sum")
    ).reset_index()
    out["Response_Rate_%"] = (out["Responded"] / out["Customers"] * 100).round(2)
    return out

## Task 1 - Gender vs response

In [9]:
gender_table = response_table(df, "Gender")
gender_table

,Gender,Customers,Responded,Response_Rate_%
0,Female,6007,2629,43.77
1,Male,3993,1361,34.08


## Task 2 - Age group vs response

In [10]:
age_table = response_table(df, "AgeGroup")
age_table

,AgeGroup,Customers,Responded,Response_Rate_%
0,<30,2499,823,32.93
1,30-50,4992,2349,47.06
2,>50,2509,818,32.60


## Task 3 - Purchase last quarter vs response

In [11]:
purchase_table = response_table(df, "Purchase_Last_Quarter")
purchase_table

,Purchase_Last_Quarter,Customers,Responded,Response_Rate_%
0,No,3486,756,21.69
1,Yes,6514,3234,49.65


## Task 4 - Product usage vs response

In [12]:
product_table = response_table(df, "ProductGroup")
product_table

,ProductGroup,Customers,Responded,Response_Rate_%
0,1-4,2091,378,18.08
1,5-8,3703,1425,38.48
2,>8,4206,2187,52.00


## Which segments respond best

3,990 of the 10,000 customers responded, so the baseline is **39.9%**.

| Segment | Best group | Rate | Worst group | Rate | Spread |
|---|---|---|---|---|---|
| Products bought | >8 | 52.00% | 1-4 | 18.08% | 33.9 pts |
| Bought last quarter | Yes | 49.65% | No | 21.69% | 28.0 pts |
| Age group | 30-50 | 47.06% | >50 | 32.60% | 14.5 pts |
| Gender | Female | 43.77% | Male | 34.08% | 9.7 pts |

Behaviour matters much more than demographics. The best customers to target are those who bought last quarter and have tried more than 8 products. The 30-50 age band and female customers also respond better, so those could be a secondary filter. Age isn't linear, since the middle band is the best and both ends are about the same.

These are single-variable breakdowns, so the effects could overlap. Modelling them together, for example with logistic regression, would show how much each one adds on its own.